In [1]:
import os
from typing import Dict
from pathlib import Path

import torch
from filelock import FileLock
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.transforms import Normalize, ToTensor
from torchvision.models import VisionTransformer
from tqdm import tqdm

TEST_ROOT = Path(os.path.expanduser("~/ray-tutorial/train"))
DATA_DIR = TEST_ROOT / "data"
BEEGFS_STORAGE_PATH = Path("/mnt/beegfs/ray-test/train-tutorial")

In [2]:
print(DATA_DIR)

/home/nico/ray-tutorial/train/data


In [3]:

def get_dataloaders(batch_size):
    # Transform to normalize the input images.
    transform = transforms.Compose([ToTensor(), Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])


    with FileLock(os.path.expanduser(DATA_DIR / "train.lock")):
        # Download training data from open datasets.
        training_data = datasets.CIFAR10(
            root=DATA_DIR,
            train=True,
            download=True,
            transform=transform,
        )

        # Download test data from open datasets.
        testing_data = datasets.CIFAR10(
            root=DATA_DIR,
            train=False,
            download=True,
            transform=transform,
        )

    # Create data loaders.
    train_dataloader = DataLoader(training_data, batch_size=batch_size, shuffle=True)
    test_dataloader = DataLoader(testing_data, batch_size=batch_size)

    return train_dataloader, test_dataloader

In [33]:
# train_dataloader, valid_dataloader = get_dataloaders(batch_size=512)

In [34]:
# X, y = next(iter(train_dataloader))

In [35]:
# import matplotlib.pyplot as plt
# plt.imshow(X[5].permute(1, 2, 0)), y[5]

In [36]:
# y.shape[0]

In [4]:
def train_func():
    lr = 1e-3
    epochs = 1
    batch_size = 512

    # Get data loaders inside the worker training function.
    train_dataloader, valid_dataloader = get_dataloaders(batch_size=batch_size)

    model = VisionTransformer(
        image_size=32,   # CIFAR-10 image size is 32x32
        patch_size=4,    # Patch size is 4x4
        num_layers=12,   # Number of transformer layers
        num_heads=8,     # Number of attention heads
        hidden_dim=384,  # Hidden size (can be adjusted)
        mlp_dim=768,     # MLP dimension (can be adjusted)
        num_classes=10   # CIFAR-10 has 10 classes
    )
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)

    # Model training loop.
    for epoch in range(epochs):
        model.train()
        for X, y in tqdm(train_dataloader, desc=f"Train Epoch {epoch}"):
            X, y = X.to(device), y.to(device)
            pred = model(X)
            loss = loss_fn(pred, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        model.eval()
        valid_loss, num_correct, num_total = 0, 0, 0
        with torch.no_grad():
            for X, y in tqdm(valid_dataloader, desc=f"Valid Epoch {epoch}"):
                X, y = X.to(device), y.to(device)
                pred = model(X)
                loss = loss_fn(pred, y)

                valid_loss += loss.item()
                num_total += y.shape[0]
                num_correct += (pred.argmax(1) == y).sum().item()

        valid_loss /= len(train_dataloader)
        accuracy = num_correct / num_total

        print({"epoch_num": epoch, "loss": valid_loss, "accuracy": accuracy})

In [5]:
%%time
train_func()

/home/nico/miniconda3/envs/drp/lib/python3.14/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")
Valid Epoch 0: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:03<00:00,  6.65it/s]

{'epoch_num': 0, 'loss': 0.38729282058015163, 'accuracy': 0.2761}
CPU times: user 36.3 s, sys: 839 ms, total: 37.2 s
Wall time: 37 s


## Distributed

In [6]:
import ray.train
from ray.train import RunConfig, ScalingConfig
from ray.train.torch import TorchTrainer

import tempfile
import uuid

In [7]:
def train_func_per_worker(config: dict):
    lr = config["lr"] 
    epochs = config["epochs"]
    batch_size = config["batch_size_per_worker"]

    # Get data loaders inside the worker training function.
    train_dataloader, valid_dataloader = get_dataloaders(batch_size=batch_size)
    train_dataloader = ray.train.torch.prepare_data_loader(train_dataloader)
    valid_dataloader = ray.train.torch.prepare_data_loader(valid_dataloader)

    model = VisionTransformer(
        image_size=32,   # CIFAR-10 image size is 32x32
        patch_size=4,    # Patch size is 4x4
        num_layers=12,   # Number of transformer layers
        num_heads=8,     # Number of attention heads
        hidden_dim=384,  # Hidden size (can be adjusted)
        mlp_dim=768,     # MLP dimension (can be adjusted)
        num_classes=10   # CIFAR-10 has 10 classes
    )
    model = ray.train.torch.prepare_model(model)

    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)

    # Model training loop.
    for epoch in range(epochs):
        if ray.train.get_context().get_world_rank() > 1:
            train_dataloader.sampler.set_epoch(epoch)
        model.train()
        for X, y in tqdm(train_dataloader, desc=f"Train Epoch {epoch}"):
            pred = model(X)
            loss = loss_fn(pred, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        model.eval()
        valid_loss, num_correct, num_total = 0, 0, 0
        with torch.no_grad():
            for X, y in tqdm(valid_dataloader, desc=f"Valid Epoch {epoch}"):
                pred = model(X)
                loss = loss_fn(pred, y)
                
                valid_loss += loss.item()
                num_correct += (pred.argmax(1) == y).sum().item()
                num_total += len(pred)

        valid_loss /= len(train_dataloader)
        accuracy = num_correct / num_total

        if ray.train.get_context().get_world_rank() == 0:
            print({"epoch_num": epoch, "loss": valid_loss, "accuracy": accuracy})
        with tempfile.TemporaryDirectory() as temp_checkpoint_dir:
            torch.save(
                model.module.state_dict(),
                os.path.join(temp_checkpoint_dir, "model.pt")
            )
            ray.train.report(
                metrics={"loss": valid_loss, "accuracy": accuracy},
                checkpoint=ray.train.Checkpoint.from_directory(temp_checkpoint_dir),
            )
            if ray.train.get_context().get_world_rank() == 0:
                print({"epoch_num": epoch, "loss": valid_loss, "accuracy": accuracy})

In [8]:
def train_cifar_10(num_workers, use_gpu):
    global_batch_size = 512

    train_config = {
        "lr": 1e-3,
        "epochs": 1,
        "batch_size_per_worker": global_batch_size // num_workers,
    }

    # [1] Start distributed training.
    # Define computation resources for workers.
    # Run `train_func_per_worker` on those workers.
    # =============================================
    scaling_config = ScalingConfig(
        num_workers=num_workers,
        use_gpu=use_gpu,
        resources_per_worker={
            "CPU": 4,
            "GPU": 1,
        }
    )
    run_config = RunConfig(
        # /mnt/cluster_storage is an Anyscale-specific storage path.
        # OSS users should set up this path themselves.
        storage_path=BEEGFS_STORAGE_PATH, 
        name=f"train_run-{uuid.uuid4().hex}",
    )
    trainer = TorchTrainer(
        train_loop_per_worker=train_func_per_worker,
        train_loop_config=train_config,
        scaling_config=scaling_config,
        run_config=run_config,
    )
    result = trainer.fit()
    print(f"Training result: {result}")

In [9]:
%%time
train_cifar_10(num_workers=4, use_gpu=True)

2026-05-30 00:26:52,881	INFO worker.py:1814 -- Connecting to existing Ray cluster at address: 10.0.1.14:6379...
2026-05-30 00:26:52,911	INFO worker.py:2003 -- Connected to Ray cluster. View the dashboard at http://127.0.0.1:8265 
/home/nico/miniconda3/envs/drp/lib/python3.14/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(
(TrainController pid=2118918) Requesting resources: {'CPU': 4, 'GPU': 1} * 4
(TrainController pid=2118918) Attempting to start training worker group of size 4 with the following resources: [{'CPU': 4, 'GPU': 1}] * 4
(RayTrainWorker pid=2119004) Setting up process group for: env:// [rank=0, world_size=4]
(PlacementGroupCleaner pid=1598303, ip=10.0.1.38) Failed to query Ray Train Controller actor state. State AP

Training result: Result(metrics={'loss': 0.3572559040419909, 'accuracy': 0.3404}, checkpoint=Checkpoint(filesystem=local, path=/mnt/beegfs/ray-test/train-tutorial/train_run-9a1b236148e44badafe0870a8b64d154/checkpoint_2026-05-30_00-28-22.281750), error=None, path='/mnt/beegfs/ray-test/train-tutorial/train_run-9a1b236148e44badafe0870a8b64d154', metrics_dataframe=       loss  accuracy
0  0.357256    0.3404, best_checkpoints=[(Checkpoint(filesystem=local, path=/mnt/beegfs/ray-test/train-tutorial/train_run-9a1b236148e44badafe0870a8b64d154/checkpoint_2026-05-30_00-28-22.281750), {'loss': 0.3572559040419909, 'accuracy': 0.3404})], _storage_filesystem=<pyarrow._fs.LocalFileSystem object at 0x7f26a4222a30>)
CPU times: user 1.04 s, sys: 150 ms, total: 1.19 s
Wall time: 1min 35s


(PlacementGroupCleaner pid=1598303, ip=10.0.1.38) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.


In [6]:
# num_workers = 2
# use_gpu = True

# global_batch_size = 512

# train_config = {
#     "lr": 1e-3,
#     "epochs": 1,
#     "batch_size_per_worker": global_batch_size // num_workers,
# }

# # [1] Start distributed training.
# # Define computation resources for workers.
# # Run `train_func_per_worker` on those workers.
# # =============================================
# scaling_config = ScalingConfig(
#     num_workers=num_workers,
#     use_gpu=use_gpu,
#     resources_per_worker={
#         "CPU": 4,
#         "GPU": 1,
#     }
# )
# run_config = RunConfig(
#     # /mnt/cluster_storage is an Anyscale-specific storage path.
#     # OSS users should set up this path themselves.
#     storage_path=DATA_DIR, 
#     name=f"train_run-{uuid.uuid4().hex}",
# )
# trainer = TorchTrainer(
#     train_loop_per_worker=train_func_per_worker,
#     train_loop_config=train_config,
#     scaling_config=scaling_config,
#     run_config=run_config,
# )
# result = trainer.fit()

In [1]:
# import os
# import tempfile

# import torch
# from torch.nn import CrossEntropyLoss
# from torch.optim import AdamW
# from torch.utils.data import DataLoader
# from torchvision.models import resnet50
# from torchvision.datasets import FashionMNIST
# from torchvision.transforms import ToTensor, Normalize, Compose

# import ray.train.torch

# 

In [2]:
# def train_func():
#     # Model, Loss, Optimizer
#     model = resnet50(num_classes=10)
#     model.conv1 = torch.nn.Conv2d(
#         1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
#     )
#     # [1] Prepare model.
#     model = ray.train.torch.prepare_model(model)
#     # model.to("cuda")  # This is done by `prepare_model`
#     criterion = CrossEntropyLoss()
#     optimizer = AdamW(model.parameters(), lr=0.001)

#     # Data
#     transform = Compose([ToTensor(), Normalize((0.28604,), (0.32025,))])
#     data_dir = os.path.join(tempfile.gettempdir(), "data")
#     train_data = FashionMNIST(root=data_dir, train=True, download=True, transform=transform)
#     train_loader = DataLoader(train_data, batch_size=4096, shuffle=True, num_workers=8, pin_memory=True, persistent_workers=True)
#     # [2] Prepare dataloader.
#     train_loader = ray.train.torch.prepare_data_loader(train_loader)

#     # Training
#     for epoch in range(50):
#         if ray.train.get_context().get_world_size() > 1:
#             train_loader.sampler.set_epoch(epoch)

#         for images, labels in train_loader:
#             # This is done by `prepare_data_loader`!
#             # images, labels = images.to("cuda"), labels.to("cuda")
#             outputs = model(images)
#             loss = criterion(outputs, labels)
#             optimizer.zero_grad()
#             loss.backward()
#             optimizer.step()

#         # [3] Report metrics and checkpoint.
#         metrics = {"loss": loss.item(), "epoch": epoch}
#         if epoch % 10 == 0:
#             with tempfile.TemporaryDirectory() as temp_checkpoint_dir:
#                 unwrap_model = getattr(model, "module", model)
#                 torch.save(
#                     unwrap_model.state_dict(),
#                     os.path.join(temp_checkpoint_dir, "model.pt")
#                 )
#                 # os.chmod(os.path.join(temp_checkpoint_dir, "model.pt"), 0o777)
#                 ray.train.report(
#                     metrics,
#                     checkpoint=ray.train.Checkpoint.from_directory(temp_checkpoint_dir),
#                 )
#         if ray.train.get_context().get_world_rank() == 0:
#             print(metrics)

# # [4] Configure scaling and resource requirements.
# scaling_config = ray.train.ScalingConfig(
#     num_workers=2,
#     use_gpu=True,
#     resources_per_worker={
#         "CPU": 8,
#         "GPU": 1.0, # Assigns half a GPU per training worker
#     }
# )

# # [5] Launch distributed training job.
# trainer = ray.train.torch.TorchTrainer(
#     train_func,
#     scaling_config=scaling_config,
#     # [5a] If running in a multi-node cluster, this is where you
#     # should configure the run's persistent storage that is accessible
#     # across all worker nodes.
#     # run_config=ray.train.RunConfig(storage_path=BEEGFS_STORAGE_PATH),
# )
# result = trainer.fit()

# # [6] Load the trained model.
# with result.checkpoint.as_directory() as checkpoint_dir:
#     model_state_dict = torch.load(os.path.join(checkpoint_dir, "model.pt"))
#     model = resnet50(num_classes=10)
#     model.conv1 = torch.nn.Conv2d(
#         1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
#     )
#     model.load_state_dict(model_state_dict)

2026-05-29 18:24:35,150	INFO worker.py:1814 -- Connecting to existing Ray cluster at address: 10.0.1.14:6379...
2026-05-29 18:24:35,180	INFO worker.py:2003 -- Connected to Ray cluster. View the dashboard at http://127.0.0.1:8265 
/home/nico/miniconda3/envs/drp/lib/python3.14/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(
(TrainController pid=2066345) Requesting resources: {'CPU': 8, 'GPU': 1.0} * 2
(TrainController pid=2066345) Attempting to start training worker group of size 2 with the following resources: [{'CPU': 8, 'GPU': 1.0}] * 2
(RayTrainWorker pid=2066415) Setting up process group for: env:// [rank=0, world_size=2]
(TrainController pid=2066345) Started training worker group of size 2: 
(TrainController pid=2066345) - 

SystemExit: 1

/home/nico/miniconda3/envs/drp/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3755: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
